In [ ]:
import numpy as np



r0 = 0.05 # term structure flat at 5% at t = 0
K = 0.05 # strike
k_s, k_x = 4.0, 8.0
Ts, Tx = 1, 2
tau = 0.25
N = 16
T = [tau*k for k in range(N+1)]
sigma = np.zeros(N+1)
sigma[1:8], sigma[8:12], sigma[12:17] = 0.2, 0.22, 0.24

def g(k, t):
    return min(1, max(T[k] - t, 0) / tau)

def theta(k):
    return (np.pi * (k-1)) / 30

def rho(i, j):
    return np.cos(np.pi * (i - j) / 30)

def mu(k, step, R_row):
    t = T[step]
    return sigma[k] * g(k, t) * np.sum([(tau * rho(j, k) * sigma[j] * R_row[j]) / (1 + tau * R_row[j]) for j in range(1, k + 1)])

def D(e_step, R):
    d = 1.0
    for j in range(1, e_step+1):
        d /= (1 + tau * R[j, j])   
    return d

def swap_receiver(step, R):
    value = 0.0 
    disc = 1.0
    for k in range(step+1, N+1):         
        disc /= (1 + tau*R[step, k])     
        value += tau*disc*(K - R[step, k])
    return value


def simulate_R(n_paths):
    
    n_steps = int(Tx / tau)          # ex: Tx=2 -> 8 steps : T_0,...,T_8
    R = np.zeros((n_paths, n_steps+1, N+1))
    
    # R_k(0) for k=1...N 
    for p in range(n_paths):
        for k in range(1, N+1):
            R[p, 0, k] = 1/tau*(np.exp(tau*r0)-1)

    # Brownian increments 
    dW1 = np.sqrt(tau) * np.random.randn(n_paths, n_steps)
    dW2 = np.sqrt(tau) * np.random.randn(n_paths, n_steps)

   
    for p in range(n_paths):
        for step in range(1, n_steps+1):
            t_prev = T[step-1] 
            for k in range(1, N+1):
                # if T_step > T_k , forward hs "expired"
                if step > k:
                    R[p, step, k] = R[p, step-1, k]
                    continue

                
                dw = (np.cos(theta(k)) * dW1[p, step-1] +
                      np.sin(theta(k)) * dW2[p, step-1])

                # drift 
                mu_k = mu(k, step-1, R[p, step-1, :])  

                #vol
                vol = sigma[k] * g(k, t_prev)

                if step == k: #last period formula is different
                    drift = tau * (mu_k - vol**2 / 6)
                    diff  = vol * dw / np.sqrt(3)
                else:
                    drift = tau * (mu_k - vol**2 / 2)
                    diff  = vol * dw

                R[p, step, k] = R[p, step-1, k] * np.exp(drift + diff)

    return R

def andersen_training(H_grid, n_paths=2000):

    # Exercise steps : 4,5,6,7,8
    exercise_steps = [int(Ts/tau) + i for i in range(int((Tx-Ts)/tau)+1)]


    Rpaths = simulate_R(n_paths)

    thresholds = {}
    thresholds[exercise_steps[-1]] = 0.0  # last date threshold = 0

    # backward loop
    for step in reversed(exercise_steps[:-1]):
        relevant_steps = [s for s in exercise_steps if s >= step]
        best_H = None
        best_value = -1e18

        for H in H_grid:
            payoffs = np.zeros(n_paths)

            for p in range(n_paths):
                for e in relevant_steps:

                    swap = swap_receiver(e, Rpaths[p])

                    H_eff = H if e == step else thresholds[e]

                    if swap > H_eff:
                        payoffs[p] = D(e, Rpaths[p]) * swap
                        break

            value = np.mean(payoffs)
            if value > best_value:
                best_value = value
                best_H = H

        thresholds[step] = best_H

    return thresholds

def andersen_pricing(thresholds, n_paths=5000):
    exercise_steps = [int(Ts/tau) + i for i in range(int((Tx-Ts)/tau)+1)]
    Rpaths = simulate_R(n_paths)

    payoffs = np.zeros(n_paths)

    for p in range(n_paths):
        for e in exercise_steps:
            swap = swap_receiver(e, Rpaths[p])
            if swap > thresholds[e]:  
                payoffs[p] = D(e, Rpaths[p]) * swap
                break

    price = np.mean(payoffs)
    se    = np.std(payoffs)/np.sqrt(n_paths)
    return price, se

H_grid = np.linspace(0.0, 0.05, 100)

thresholds = andersen_training(H_grid, n_paths=10000)
price, se = andersen_pricing(thresholds, n_paths=20000)

print("Seuils optimaux :")
for step, H in thresholds.items():
    print(f"  T = {T[step]:.2f}  ->  H* = {H:.6f}")

print(f"Prix Bermudan : {price:.6f}  +/- {se:.6f}")

Seuils optimaux :
  T = 2.00  ->  H* = 0.000000
  T = 1.75  ->  H* = 0.011616
  T = 1.50  ->  H* = 0.013636
  T = 1.25  ->  H* = 0.018182
  T = 1.00  ->  H* = 0.021212
Prix Bermudan : 0.013538  +/- 0.000104
